In [1]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Context Graph Demo V2: System of Reasoning for Agentic Ads

**Integrating BigQuery Agent Analytics SDK with Native Property Graphs**

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/context_graph_adcp_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/colab-logo.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/main/examples/context_graph_adcp_demo.ipynb">
      <img src="https://www.gstatic.com/images/branding/product/1x/google_cloud_48dp.png" alt="Vertex AI logo" width="32"> Open in Vertex AI Workbench
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/context_graph_adcp_demo.ipynb">
      <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTW1gvOovVlbZAIZylUtf5Iu8-693qS1w5NJw&s" alt="BQ logo" width="35"> Open in BQ Studio
    </a>
  </td>
</table>

## Demo Scenario: Yahoo ADCP "ELF Cosmetics" Media Buy

This notebook simulates a realistic **multi-agent media buying workflow** based on the Yahoo Ad Context Protocol (ADCP):

```mermaid
flowchart LR
    A[Buyer Agent<br>ELF Cosmetics] -->|ADCP Brief| B[Sales Agent<br>Yahoo DSP]
    B -->|Inventory Query| C[Inventory Tool]
    B -->|Audience Match| D[Audience Tool]
    B -->|Budget Split| E[Budget Tool]
    B -->|HITL Pause| F[Ad Ops Manager<br>Slack Approval]
    F -->|Approved| G[Provision Campaign<br>Google Ad Manager]
    G -->|Artifact| H[GCS Line Item JSON]
```

We then build a **4-Pillar Context Graph** that cross-links:
1. **Technical Graph** (execution lineage from ADK plugin)
2. **Biz Graph** (Products, Targeting, Campaigns extracted via AI.GENERATE)
3. **Cross-Links** (the "Missing Why" connecting decisions to entities)
4. **Persisted Artifacts** (GCS object references for campaign JSON)

Finally, we demonstrate **World Change detection** for long-running A2A tasks.

## Install Dependencies

In [2]:
!pip install -q google-adk bigquery-agent-analytics google-cloud-bigquery nest-asyncio

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621


ERROR: Ignored the following versions that require a different python version: 1.19.0 Requires-Python >=3.10; 1.20.0 Requires-Python >=3.10; 1.21.0 Requires-Python >=3.10; 1.22.0 Requires-Python >=3.10; 1.22.1 Requires-Python >=3.10; 1.23.0 Requires-Python >=3.10; 1.24.0 Requires-Python >=3.10; 1.24.1 Requires-Python >=3.10; 1.25.0 Requires-Python >=3.10; 1.25.1 Requires-Python >=3.10; 1.26.0 Requires-Python >=3.10
ERROR: Could not find a version that satisfies the requirement bigquery-agent-analytics (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /usr/local/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip
ERROR: No matching distribution found for bigquery-agent-analytics


## Authenticate & Configure

In [3]:
import os

try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab -- using default credentials.")

# ---------- Configuration ----------
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
DATASET_ID = os.environ.get("BQ_DATASET", "agent_analytics")
TABLE_ID = os.environ.get("BQ_TABLE", "agent_events")
MODEL_NAME = os.environ.get("MODEL_NAME", "gemini-2.5-flash")
LOCATION = "US"
APP_NAME = "adcp_context_graph_demo"
USER_ID = "adcp_demo_user"

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

import nest_asyncio
nest_asyncio.apply()

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID}")
print(f"Table    : {TABLE_ID}")
print(f"Model    : {MODEL_NAME}")
print(f"Vertex AI: enabled")

Not running in Colab -- using default credentials.
Project  : test-project-0728-467323
Dataset  : agent_analytics
Table    : agent_events
Model    : gemini-3-flash-preview
Vertex AI: enabled


---

## Phase 1: Define ADCP Domain Tools

We create **deterministic tools** that simulate Yahoo's advertising platform. Each tool uses seeded randomness for reproducible demo output.

In [4]:
import hashlib
import json
import random
from typing import Any


def _rng_from(*parts: str) -> random.Random:
    seed = int(hashlib.md5("|".join(parts).encode()).hexdigest()[:8], 16)
    return random.Random(seed)


async def query_ad_inventory(
    product_name: str,
    format_type: str = "display",
    date_range: str = "2025-Q2",
) -> dict[str, Any]:
    """Query available ad inventory for a Yahoo product.

    Args:
        product_name: Yahoo ad product (e.g. 'Yahoo Homepage', 'Yahoo Mail').
        format_type: Ad format ('display', 'native', 'video').
        date_range: Date range for availability.

    Returns:
        Inventory availability with pricing.
    """
    rng = _rng_from(product_name, format_type, date_range)
    available_impressions = rng.randint(500_000, 50_000_000)
    cpm = round(rng.uniform(3.50, 45.00), 2)
    return {
        "product": product_name,
        "format": format_type,
        "date_range": date_range,
        "available_impressions": available_impressions,
        "cpm_usd": cpm,
        "estimated_reach": rng.randint(100_000, 10_000_000),
        "viewability_rate": round(rng.uniform(0.60, 0.95), 2),
        "status": rng.choice(["available", "available", "available", "limited"]),
        "premium_placement": product_name in ("Yahoo Homepage", "Yahoo Finance"),
    }


async def match_target_audience(
    brand: str,
    target_demographics: str,
    campaign_goal: str = "brand_awareness",
) -> dict[str, Any]:
    """Match target audience segments against Yahoo's audience graph.

    Args:
        brand: Advertiser brand name.
        target_demographics: Target audience description.
        campaign_goal: Campaign objective.

    Returns:
        Matched audience segments with reach estimates.
    """
    rng = _rng_from(brand, target_demographics)
    segments = [
        {"segment": "Beauty Enthusiasts", "match_score": 0.95},
        {"segment": "Millennials 25-34", "match_score": 0.92},
        {"segment": "Female Shoppers", "match_score": 0.88},
        {"segment": "Health & Wellness", "match_score": 0.76},
        {"segment": "Premium Consumers", "match_score": 0.71},
    ]
    rng.shuffle(segments)
    selected = segments[:rng.randint(3, 5)]
    return {
        "brand": brand,
        "target_demographics": target_demographics,
        "campaign_goal": campaign_goal,
        "matched_segments": sorted(selected, key=lambda s: -s["match_score"]),
        "total_addressable_audience": rng.randint(2_000_000, 15_000_000),
        "overlap_with_yahoo_users_pct": round(rng.uniform(0.45, 0.85), 2),
    }


async def allocate_media_budget(
    total_budget_usd: float,
    products: str,
    campaign_duration_days: int = 30,
) -> dict[str, Any]:
    """Allocate media budget across Yahoo ad products.

    Args:
        total_budget_usd: Total campaign budget in USD.
        products: Comma-separated list of Yahoo products.
        campaign_duration_days: Campaign length in days.

    Returns:
        Budget allocation with ROI projections.
    """
    product_list = [p.strip() for p in products.split(",")]
    rng = _rng_from(str(total_budget_usd), products)
    allocations = []
    remaining = total_budget_usd

    for i, product in enumerate(product_list):
        if i == len(product_list) - 1:
            amount = round(remaining, 2)
        else:
            pct = rng.uniform(0.15, 0.50)
            amount = round(total_budget_usd * pct, 2)
            remaining -= amount

        allocations.append({
            "product": product,
            "allocated_usd": amount,
            "pct_of_total": round(amount / total_budget_usd * 100, 1),
            "estimated_impressions": int(amount / rng.uniform(5, 25) * 1000),
            "projected_ctr_pct": round(rng.uniform(0.8, 3.5), 2),
        })

    return {
        "total_budget_usd": total_budget_usd,
        "campaign_duration_days": campaign_duration_days,
        "allocations": allocations,
        "projected_total_impressions": sum(
            a["estimated_impressions"] for a in allocations
        ),
        "projected_roas": round(rng.uniform(2.5, 8.0), 2),
    }


async def provision_campaign_in_gam(
    campaign_name: str,
    advertiser: str,
    budget_usd: float,
    start_date: str,
    end_date: str,
    targeting_segments: str,
) -> dict[str, Any]:
    """Provision a campaign in Google Ad Manager (GAM).

    Args:
        campaign_name: Name for the campaign.
        advertiser: Advertiser name.
        budget_usd: Total budget.
        start_date: Campaign start date.
        end_date: Campaign end date.
        targeting_segments: Comma-separated targeting segments.

    Returns:
        Provisioned campaign details with line item IDs.
    """
    rng = _rng_from(campaign_name, advertiser)
    line_item_id = f"LI-{rng.randint(100000, 999999)}"
    order_id = f"ORD-{rng.randint(10000, 99999)}"

    artifact = {
        "campaign_name": campaign_name,
        "advertiser": advertiser,
        "order_id": order_id,
        "line_item_id": line_item_id,
        "budget_usd": budget_usd,
        "start_date": start_date,
        "end_date": end_date,
        "targeting": [s.strip() for s in targeting_segments.split(",")],
        "status": "provisioned",
        "gcs_artifact_uri": (
            f"gs://adcp-artifacts/{advertiser.lower().replace(' ', '_')}/"
            f"{line_item_id}.json"
        ),
    }
    return artifact


print("ADCP tools defined: query_ad_inventory, match_target_audience,"
      " allocate_media_budget, provision_campaign_in_gam")

ADCP tools defined: query_ad_inventory, match_target_audience, allocate_media_budget, provision_campaign_in_gam


---

## Phase 2: Build Multi-Agent System with ADK

We create a **Yahoo Sales Agent** that orchestrates the ADCP workflow. It processes the buyer's brief, queries inventory, matches audiences, allocates budget, and pauses for HITL approval before provisioning.

In [5]:
from google.adk.agents import LlmAgent
from google.genai import types

YAHOO_SALES_AGENT_INSTRUCTION = """\
You are the Yahoo DSP Sales Agent operating under the Ad Context Protocol (ADCP).
You process media buying requests from buyer agents and human ad planners.

Workflow for processing a media buying brief:
1. Parse the buyer's brief to identify: brand, target demographics, budget, and campaign goals.
2. Query ad inventory for relevant Yahoo products (Yahoo Homepage, Yahoo Mail, Yahoo Finance, Yahoo Sports).
3. Match the target audience against Yahoo's audience graph.
4. Allocate the media budget across recommended products.
5. Present the media plan for human review.
6. Once approved, provision the campaign in Google Ad Manager (GAM).

Guidelines:
- Always query at least 2 Yahoo products for inventory.
- Recommend budget allocation based on audience match scores.
- Flag any premium placements (Yahoo Homepage, Yahoo Finance).
- Present clear reasoning for each recommendation.
- Include projected ROI and impression estimates.
"""


def build_adcp_agent() -> LlmAgent:
    """Build the Yahoo ADCP Sales Agent."""
    return LlmAgent(
        name="yahoo_sales_agent",
        model=MODEL_NAME,
        instruction=YAHOO_SALES_AGENT_INSTRUCTION,
        tools=[
            query_ad_inventory,
            match_target_audience,
            allocate_media_budget,
            provision_campaign_in_gam,
        ],
        generate_content_config=types.GenerateContentConfig(
            temperature=1.0,
        ),
    )


print("ADCP agent builder ready.")

/usr/local/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


ADCP agent builder ready.


---

## Phase 3: Run ADCP Workflows & Log Traces to BigQuery

We run **three simulated ADCP conversations** representing different media buying scenarios. The `BigQueryAgentAnalyticsPlugin` captures every event.

In [6]:
import asyncio
import uuid

from google.adk.plugins.bigquery_agent_analytics_plugin import (
    BigQueryAgentAnalyticsPlugin,
    BigQueryLoggerConfig,
)
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

agent = build_adcp_agent()
session_service = InMemorySessionService()

plugin = BigQueryAgentAnalyticsPlugin(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    config=BigQueryLoggerConfig(
        table_id=TABLE_ID,
        batch_size=1,
        batch_flush_interval=1.0,
    ),
    location=LOCATION,
)

runner = Runner(
    agent=agent,
    app_name=APP_NAME,
    session_service=session_service,
    plugins=[plugin],
)

# ---------- Three ADCP Conversations ----------
conversations = [
    {
        "label": "ELF Cosmetics -- $50K Brand Awareness Campaign",
        "messages": [
            (
                "ADCP Media Buying Brief:\n"
                "Brand: ELF Cosmetics\n"
                "Budget: $50,000\n"
                "Campaign Goal: Brand awareness for new skincare line\n"
                "Target Demographics: Millennials 25-34, beauty enthusiasts\n"
                "Flight Dates: 2025-05-01 to 2025-05-31\n"
                "Preferred Products: Yahoo Homepage, Yahoo Mail\n\n"
                "Please process this brief: query inventory for Yahoo Homepage "
                "and Yahoo Mail, match the target audience, allocate the $50,000 "
                "budget across the recommended products, and present the media plan."
            ),
            (
                "The media plan is approved by the ad-ops manager. "
                "Please provision the campaign in Google Ad Manager with "
                "campaign name 'ELF_Skincare_May2025', advertiser 'ELF Cosmetics', "
                "budget $50000, start date 2025-05-01, end date 2025-05-31, "
                "and targeting segments 'Beauty Enthusiasts, Millennials 25-34'."
            ),
        ],
    },
    {
        "label": "Nike -- $200K Multi-Product Performance Campaign",
        "messages": [
            (
                "ADCP Media Buying Brief:\n"
                "Brand: Nike\n"
                "Budget: $200,000\n"
                "Campaign Goal: Product launch for Air Max 2025\n"
                "Target Demographics: Sports enthusiasts 18-45, sneakerheads\n"
                "Flight Dates: 2025-06-01 to 2025-06-30\n"
                "Preferred Products: Yahoo Sports, Yahoo Homepage, Yahoo Finance\n\n"
                "Process this brief: check inventory for Yahoo Sports, "
                "Yahoo Homepage, and Yahoo Finance. Match target audiences "
                "and recommend a budget split across all three products."
            ),
        ],
    },
    {
        "label": "Tesla -- $100K Targeted EV Campaign",
        "messages": [
            (
                "ADCP Media Buying Brief:\n"
                "Brand: Tesla\n"
                "Budget: $100,000\n"
                "Campaign Goal: Lead generation for Model Y test drives\n"
                "Target Demographics: Tech-savvy professionals 30-55, EV intenders\n"
                "Flight Dates: 2025-07-01 to 2025-07-31\n"
                "Preferred Products: Yahoo Finance, Yahoo Mail\n\n"
                "Query inventory for Yahoo Finance and Yahoo Mail, "
                "match the target audience, and allocate the budget. "
                "Then provision the campaign in GAM with name 'Tesla_ModelY_Jul2025', "
                "advertiser 'Tesla', budget $100000, start date 2025-07-01, "
                "end date 2025-07-31, targeting 'Tech Professionals, EV Intenders'."
            ),
        ],
    },
]


async def run_conversation(messages, label=""):
    """Run a multi-turn ADCP conversation."""
    session_id = f"adcp-{uuid.uuid4().hex[:12]}"
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
    )
    print(f"\n{'=' * 70}")
    print(f"  Session: {session_id}  [{label}]")
    print(f"{'=' * 70}")

    for i, message in enumerate(messages, 1):
        print(f"\n[Turn {i}] Buyer: {message[:120]}...")
        print("-" * 48)
        user_content = types.Content(
            role="user",
            parts=[types.Part(text=message)],
        )
        response_parts = []
        async for event in runner.run_async(
            user_id=USER_ID,
            session_id=session_id,
            new_message=user_content,
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if hasattr(part, "text") and part.text:
                        response_parts.append(part.text)
                    elif hasattr(part, "function_call") and part.function_call:
                        print(f"  -> Tool call: {part.function_call.name}")
        if response_parts:
            text = "\n".join(response_parts)
            print(f"\n[Sales Agent]: {text[:800]}")
            if len(text) > 800:
                print(f"  ... (truncated, {len(text)} chars total)")
    return session_id


# Run all conversations
session_ids = []
for conv in conversations:
    sid = asyncio.get_event_loop().run_until_complete(
        run_conversation(conv["messages"], label=conv["label"])
    )
    session_ids.append(sid)

print(f"\n\nSession IDs: {session_ids}")


  Session: adcp-a20d176b82af  [ELF Cosmetics -- $50K Brand Awareness Campaign]

[Turn 1] Buyer: ADCP Media Buying Brief:
Brand: ELF Cosmetics
Budget: $50,000
Campaign Goal: Brand awareness for new skincare line
Targe...
------------------------------------------------


  -> Tool call: query_ad_inventory
  -> Tool call: query_ad_inventory


  -> Tool call: match_target_audience


  -> Tool call: allocate_media_budget



[Sales Agent]: Based on your brief for **ELF Cosmetics**, I have developed a comprehensive media plan to drive brand awareness for your new skincare line among Millennials and beauty enthusiasts.

### **Campaign Overview**
*   **Brand:** ELF Cosmetics
*   **Campaign Goal:** Brand Awareness
*   **Total Budget:** $50,000
*   **Flight Dates:** 2025-05-01 to 2025-05-31
*   **Total Projected Impressions:** ~8,166,935
*   **Projected ROAS:** 3.55

---

### **Audience Strategy**
We matched your requirements against Yahoo’s audience graph, identifying a total addressable audience of **7.28M users**. 
*   **Primary Segment:** Beauty Enthusiasts (Match Score: 0.95)
*   **Secondary Segment:** Millennials 25-34 (Match Score: 0.92)
*   **Supporting Segment:** Female Shoppers (Match Score: 0.88)

---

### **Media Allo
  ... (truncated, 2212 chars total)

[Turn 2] Buyer: The media plan is approved by the ad-ops manager. Please provision the campaign in Google Ad Manager with campaign name ...
------

...


[Sales Agent]: Based on your brief for the **Nike Air Max 2025** product launch, I have analyzed Yahoo's inventory and audience graph to develop a recommended media plan.

### **Campaign Overview**
*   **Brand:** Nike
*   **Budget:** $200,000
*   **Flight Dates:** June 1, 2025 – June 30, 2025 (30 Days)
*   **Goal:** High-impact product launch & reach among sports/sneaker enthusiasts.

---

### **Media Plan Recommendation**

| Product | Budget Allocation | Estimated Impressions | CPM (Avg) | Projected CTR |
| :--- | :--- | :--- | :--- | :--- |
| **Yahoo Finance** | $121,327.80 (60.7%) | 5,135,264 | $19.05 | 1.21% |
| **Yahoo Homepage**| $46,134.57 (23.1%) | 2,784,630 | $21.06 | 2.35% |
| **Yahoo Sports**  | $32,537.63 (16.3%) | 1,475,740 | $5.81  | 2.25% |
| **TOTAL**         | **$200,000.00**    | **9,39
  ... (truncated, 1984 chars total)

  Session: adcp-2c401a645c40  [Tesla -- $100K Targeted EV Campaign]

[Turn 1] Buyer: ADCP Media Buying Brief:
Brand: Tesla
Budget: $100,000
Campai

  -> Tool call: query_ad_inventory
  -> Tool call: query_ad_inventory
  -> Tool call: match_target_audience


  -> Tool call: allocate_media_budget


  -> Tool call: provision_campaign_in_gam



[Sales Agent]: I have successfully processed your media buying brief for the **Tesla Model Y Lead Generation Campaign**. 

### **1. Media Plan Overview**
Based on your budget of **$100,000** and target audience of **Tech-savvy professionals and EV intenders**, I have optimized the allocation across your preferred Yahoo products.

| Yahoo Product | Format | Budget Allocation | Est. Impressions | Key Reason |
| :--- | :--- | :--- | :--- | :--- |
| **Yahoo Finance** | Display | $29,465.15 (29.5%) | 1,432,055 | **Premium Placement:** High affinity with high-net-worth tech professionals. |
| **Yahoo Mail** | Native | $70,534.85 (70.5%) | 5,808,834 | **High Engagement:** Native ads in inbox provide superior CTR (1.19%) for lead gen. |

**Campaign Performance Projections:**
*   **Total Projected Impressions:** 
  ... (truncated, 1746 chars total)


Session IDs: ['adcp-a20d176b82af', 'adcp-7d9855e7a71b', 'adcp-2c401a645c40']


In [7]:
import time

print("Flushing traces to BigQuery ...")
try:
    asyncio.get_event_loop().run_until_complete(plugin.flush())
except Exception as exc:
    print(f"Flush warning: {exc}")

settle_seconds = 15
print(f"Waiting {settle_seconds}s for BigQuery data to settle ...")
time.sleep(settle_seconds)
print("Done.")

Flushing traces to BigQuery ...
Waiting 15s for BigQuery data to settle ...


Done.


---

## Phase 4: Trace Retrieval & Visualization

Use the SDK Client to fetch traces and render the hierarchical execution DAG.

In [8]:
from bigquery_agent_analytics import Client, TraceFilter

client = Client(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    location=LOCATION,
    endpoint=MODEL_NAME,
)
print("SDK Client initialised.")

SDK Client initialised.


In [9]:
# Retrieve and render each trace
traces = []
for sid in session_ids:
    try:
        trace = client.get_session_trace(sid)
        traces.append(trace)
        print(f"\n{'=' * 70}")
        print(f"  Trace for session: {sid}")
        print(f"{'=' * 70}")
        _ = trace.render()
    except Exception as exc:
        print(f"Could not retrieve trace {sid}: {exc}")
        traces.append(None)


  Trace for session: adcp-a20d176b82af
Trace: e-7c76dd0c-74c0-4b77-80cc-074a0b546c97 | Session: adcp-a20d176b82af | 24545ms
└─ [✓] USER_MESSAGE_RECEIVED [yahoo_sales_agent] - ADCP Media Buying Brief:
Brand: ELF Cosmetics
Budget: $50,000
Campaign Goal: Brand awareness for new skincare line
Ta...
└─ [✓] INVOCATION_STARTING [yahoo_sales_agent]
└─ [✓] INVOCATION_COMPLETED [yahoo_sales_agent] (32135ms)
   ├─ [✓] AGENT_STARTING [yahoo_sales_agent] - You are the Yahoo DSP Sales Agent operating under the Ad Context Protocol (ADCP).
You process media buying requests f...
   └─ [✓] AGENT_COMPLETED [yahoo_sales_agent] (18226ms)
      ├─ [✓] LLM_REQUEST [yahoo_sales_agent] (gemini-3-flash-preview) - ADCP Media Buying Brief:
Brand: ELF Cosmetics
Budget: $50,000
Campaign Goal: Brand awareness for new skincare line
Ta...
      ├─ [✓] LLM_RESPONSE [yahoo_sales_agent] (3938ms) - call: query_ad_inventory | call: query_ad_inventory
      ├─ [✓] TOOL_STARTING [yahoo_sales_agent] (query_ad_inventory)
    


  Trace for session: adcp-7d9855e7a71b
Trace: e-3f6807e1-d3b9-4eb9-a0e3-f23dd72cee57 | Session: adcp-7d9855e7a71b | 14565ms
└─ [✓] USER_MESSAGE_RECEIVED [yahoo_sales_agent] - ADCP Media Buying Brief:
Brand: Nike
Budget: $200,000
Campaign Goal: Product launch for Air Max 2025
Target Demograph...
└─ [✓] INVOCATION_STARTING [yahoo_sales_agent]
└─ [✓] INVOCATION_COMPLETED [yahoo_sales_agent] (14564ms)
   ├─ [✓] AGENT_STARTING [yahoo_sales_agent] - You are the Yahoo DSP Sales Agent operating under the Ad Context Protocol (ADCP).
You process media buying requests f...
   └─ [✓] AGENT_COMPLETED [yahoo_sales_agent] (14564ms)
      ├─ [✓] LLM_REQUEST [yahoo_sales_agent] (gemini-3-flash-preview) - ADCP Media Buying Brief:
Brand: Nike
Budget: $200,000
Campaign Goal: Product launch for Air Max 2025
Target Demograph...
      ├─ [✓] LLM_RESPONSE [yahoo_sales_agent] (3732ms) - call: query_ad_inventory | call: query_ad_inventory | call: query_ad_inventory | call: match_target_audience
      ├─ [✓] TO


  Trace for session: adcp-2c401a645c40
Trace: e-27133476-0092-41cb-ba67-1e99d226cae5 | Session: adcp-2c401a645c40 | 15069ms
└─ [✓] USER_MESSAGE_RECEIVED [yahoo_sales_agent] - ADCP Media Buying Brief:
Brand: Tesla
Budget: $100,000
Campaign Goal: Lead generation for Model Y test drives
Target ...
└─ [✓] INVOCATION_STARTING [yahoo_sales_agent]
└─ [✓] INVOCATION_COMPLETED [yahoo_sales_agent] (15068ms)
   ├─ [✓] AGENT_STARTING [yahoo_sales_agent] - You are the Yahoo DSP Sales Agent operating under the Ad Context Protocol (ADCP).
You process media buying requests f...
   └─ [✓] AGENT_COMPLETED [yahoo_sales_agent] (15068ms)
      ├─ [✓] LLM_REQUEST [yahoo_sales_agent] (gemini-3-flash-preview) - ADCP Media Buying Brief:
Brand: Tesla
Budget: $100,000
Campaign Goal: Lead generation for Model Y test drives
Target ...
      ├─ [✓] LLM_RESPONSE [yahoo_sales_agent] (3617ms) - call: query_ad_inventory | call: query_ad_inventory | call: match_target_audience
      ├─ [✓] TOOL_STARTING [yahoo_sales_ag

In [10]:
# Inspect ADCP-specific trace properties
for i, trace in enumerate(traces):
    if trace is None:
        continue
    print(f"\n--- Session {i+1}: {trace.session_id} ---")
    print(f"  Total spans    : {len(trace.spans)}")
    print(f"  Tool calls     : {len(trace.tool_calls)}")
    for tc in trace.tool_calls:
        print(f"    - {tc.get('tool_name', '?')}")
    final = trace.final_response or "(none)"
    print(f"  Final response : {final[:300]}")
    errors = trace.error_spans
    print(f"  Error spans    : {len(errors)}")
    if trace.total_latency_ms:
        print(f"  Total latency  : {trace.total_latency_ms:.0f}ms")


--- Session 1: adcp-a20d176b82af ---
  Total spans    : 32
  Tool calls     : 5
    - query_ad_inventory
    - query_ad_inventory
    - match_target_audience
    - allocate_media_budget
    - provision_campaign_in_gam
  Final response : text: 'The campaign has been successfully provisioned in Google Ad Manager.

### **Campaign Provisioning Summary**
*   **Campaign Name:** ELF_Skincare_May2025
*   **Advertiser:** ELF Cosmetics
*   **Order ID:** `ORD-70678`
*   **Line Item ID:** `LI-756565`
*   **Budget:** $50,000.00
*   **Flight Dat
  Error spans    : 0
  Total latency  : 24545ms

--- Session 2: adcp-7d9855e7a71b ---
  Total spans    : 21
  Tool calls     : 5
    - query_ad_inventory
    - query_ad_inventory
    - query_ad_inventory
    - match_target_audience
    - allocate_media_budget
  Final response : text: 'Based on your brief for the **Nike Air Max 2025** product launch, I have analyzed Yahoo's inventory and audience graph to develop a recommended media plan.

### **Campaign Ove

---

## Phase 5: Context Graph -- Business Entity Extraction

We use BigQuery's `AI.GENERATE` with `output_schema` to extract structured business entities from the unstructured agent payloads. This creates the **Biz Graph** layer of our 4-pillar context graph.

Entity types: `Product`, `Targeting`, `Campaign`, `Budget`, `Audience`

In [11]:
from bigquery_agent_analytics import ContextGraphManager, ContextGraphConfig

# Configure the context graph for ADCP domain
# Note: For Gemini 3.x+ models, AI.GENERATE requires a full Vertex AI
# endpoint URL. The SDK handles this automatically via _resolve_endpoint().
cg_config = ContextGraphConfig(
    biz_nodes_table="adcp_biz_nodes",
    cross_links_table="adcp_cross_links",
    graph_name="adcp_context_graph",
    endpoint=MODEL_NAME,
    entity_types=[
        "Product",
        "Targeting",
        "Campaign",
        "Budget",
        "Audience",
        "Advertiser",
    ],
    max_hops=20,
)

# Option 1: Use client.context_graph() factory
cgm = client.context_graph(config=cg_config)

# Option 2: Direct instantiation
# cgm = ContextGraphManager(
#     project_id=PROJECT_ID,
#     dataset_id=DATASET_ID,
#     table_id=TABLE_ID,
#     config=cg_config,
#     location=LOCATION,
# )

print(f"ContextGraphManager ready.")
print(f"  Biz nodes table  : {cg_config.biz_nodes_table}")
print(f"  Cross-links table: {cg_config.cross_links_table}")
print(f"  Graph name       : {cg_config.graph_name}")
print(f"  Entity types     : {cg_config.entity_types}")
print(f"  AI.GENERATE endpoint: {cgm._resolve_endpoint()[:80]}...")

ContextGraphManager ready.
  Biz nodes table  : adcp_biz_nodes
  Cross-links table: adcp_cross_links
  Graph name       : adcp_context_graph
  Entity types     : ['Product', 'Targeting', 'Campaign', 'Budget', 'Audience', 'Advertiser']
  AI.GENERATE endpoint: https://aiplatform.googleapis.com/v1/projects/test-project-0728-467323/locations...


In [12]:
# Extract business entities using AI.GENERATE (server-side)
# Falls back to client-side extraction if AI.GENERATE is not available
try:
    biz_nodes = cgm.extract_biz_nodes(
        session_ids=session_ids,
        use_ai_generate=True,
    )
    print(f"Extracted {len(biz_nodes)} business entities:")
    for node in biz_nodes[:15]:
        print(f"  [{node.node_type}] {node.node_value} "
              f"(confidence={node.confidence:.2f})")
    if len(biz_nodes) > 15:
        print(f"  ... ({len(biz_nodes) - 15} more)")
except Exception as exc:
    print(f"AI.GENERATE extraction not available: {exc}")
    print("Falling back to client-side extraction ...")
    biz_nodes = cgm.extract_biz_nodes(
        session_ids=session_ids,
        use_ai_generate=False,
    )
    print(f"Fetched {len(biz_nodes)} raw payloads for manual review.")

Extracted 180 business entities:
  [Product] Yahoo Sports (confidence=1.00)
  [Advertiser] Nike (confidence=1.00)
  [Product] Yahoo Finance (confidence=1.00)
  [Advertiser] Nike (confidence=1.00)
  [Budget] 50000 (confidence=1.00)
  [Budget] 50000 (confidence=1.00)
  [Product] Yahoo Homepage (confidence=1.00)
  [Budget] 200000 (confidence=1.00)
  [Product] Yahoo Finance (confidence=1.00)
  [Product] Yahoo Mail (confidence=1.00)
  [Product] Yahoo Sports (confidence=1.00)
  [Audience] Millennials 25-34 (confidence=1.00)
  [Product] Yahoo Homepage (confidence=1.00)
  [Campaign] Tesla_ModelY_Jul2025 (confidence=1.00)
  [Budget] $50,000 (confidence=1.00)
  ... (165 more)


### Alternative: Manual Entity Extraction & Storage

If `AI.GENERATE` is not available, you can extract entities client-side and store them via `store_biz_nodes()`.

In [13]:
from bigquery_agent_analytics import BizNode

# Example: manually define business entities from the ELF campaign
manual_biz_nodes = [
    BizNode(
        span_id="manual-1",
        session_id=session_ids[0],
        node_type="Advertiser",
        node_value="ELF Cosmetics",
        confidence=1.0,
    ),
    BizNode(
        span_id="manual-2",
        session_id=session_ids[0],
        node_type="Product",
        node_value="Yahoo Homepage",
        confidence=0.95,
    ),
    BizNode(
        span_id="manual-3",
        session_id=session_ids[0],
        node_type="Product",
        node_value="Yahoo Mail",
        confidence=0.90,
    ),
    BizNode(
        span_id="manual-4",
        session_id=session_ids[0],
        node_type="Targeting",
        node_value="Millennials 25-34",
        confidence=0.92,
    ),
    BizNode(
        span_id="manual-5",
        session_id=session_ids[0],
        node_type="Targeting",
        node_value="Beauty Enthusiasts",
        confidence=0.95,
    ),
    BizNode(
        span_id="manual-6",
        session_id=session_ids[0],
        node_type="Budget",
        node_value="$50,000",
        confidence=1.0,
    ),
    BizNode(
        span_id="manual-7",
        session_id=session_ids[0],
        node_type="Campaign",
        node_value="ELF_Skincare_May2025",
        confidence=1.0,
    ),
]

print(f"Manual biz nodes prepared: {len(manual_biz_nodes)}")
for node in manual_biz_nodes:
    print(f"  [{node.node_type}] {node.node_value}")

# Uncomment to store in BigQuery:
# success = cgm.store_biz_nodes(manual_biz_nodes)
# print(f"Stored: {success}")

Manual biz nodes prepared: 7
  [Advertiser] ELF Cosmetics
  [Product] Yahoo Homepage
  [Product] Yahoo Mail
  [Targeting] Millennials 25-34
  [Targeting] Beauty Enthusiasts
  [Budget] $50,000
  [Campaign] ELF_Skincare_May2025


---

## Phase 6: Property Graph DDL

We generate and inspect the `CREATE PROPERTY GRAPH` DDL that formalizes the 4-pillar Context Graph in BigQuery. This creates:
- **TechNode** — spans from `agent_events` (execution lineage)
- **BizNode** — entities from `extracted_biz_nodes` (domain entities)
- **Caused** edges — parent→child span linkage (decision lineage)
- **Evaluated** edges — tech event → business entity cross-links

In [14]:
# Generate and display the Property Graph DDL
ddl = cgm.get_property_graph_ddl()

print("=" * 70)
print("  CREATE PROPERTY GRAPH DDL")
print("=" * 70)
print(ddl)

  CREATE PROPERTY GRAPH DDL
CREATE OR REPLACE PROPERTY GRAPH `test-project-0728-467323.agent_analytics.adcp_context_graph`
  NODE TABLES (
    -- Technical execution nodes (spans from ADK plugin)
    `test-project-0728-467323.agent_analytics.agent_events` AS TechNode
      KEY (span_id)
      LABEL TechNode
      PROPERTIES (
        event_type,
        agent,
        timestamp,
        session_id,
        invocation_id,
        content,
        latency_ms,
        status,
        error_message
      ),
    -- Business domain nodes (extracted entities)
    `test-project-0728-467323.agent_analytics.adcp_biz_nodes` AS BizNode
      KEY (node_value)
      LABEL BizNode
      PROPERTIES (
        node_type,
        node_value,
        confidence,
        session_id
      )
  )
  EDGE TABLES (
    -- Causal lineage: parent span -> child span
    `test-project-0728-467323.agent_analytics.agent_events` AS Caused
      KEY (span_id)
      SOURCE KEY (parent_span_id) REFERENCES TechNode (span_i

In [15]:
# Create the Property Graph in BigQuery
# NOTE: Property Graphs require BigQuery Studio (BQ Notebooks).
# If running outside BQ Studio, the DDL and GQL are generated for you
# to copy-paste into a BigQuery Studio notebook cell with %%bigquery magic.

# Step 1: Create cross-links
try:
    cross_links_ok = cgm.create_cross_links(session_ids)
    print(f"Cross-links created: {cross_links_ok}")
except Exception as exc:
    print(f"Cross-links creation: {exc}")

# Step 2: Create the Property Graph
try:
    graph_ok = cgm.create_property_graph()
    print(f"Property Graph created: {graph_ok}")
except Exception as exc:
    print(f"Property Graph creation: {exc}")
    print("\n--- To create the Property Graph, run the DDL above in ---")
    print("--- a BigQuery Studio notebook cell:                    ---")
    print("---   %%bigquery                                        ---")
    print("---   CREATE OR REPLACE PROPERTY GRAPH ...              ---")
    print("---                                                     ---")
    print("--- Reference: https://github.com/GoogleCloudPlatform/  ---")
    print("---   devrel-demos/blob/main/data-analytics/            ---")
    print("---   knowledge_graph_demo/kg_demo_template.ipynb       ---")

Cross-links created: True


Failed to create Property Graph: 400 Unsupported statement CREATE PROPERTY GRAPH.; reason: invalidQuery, message: Unsupported statement CREATE PROPERTY GRAPH.

Location: US
Job ID: 89c29192-bb8a-4b3a-821c-0e1d34a2ceed



Property Graph created: False


### Run in BigQuery Studio (BQ Notebooks)

Property Graphs and GQL visualization require **BigQuery Studio**. When running this notebook in BQ Studio, uncomment and execute the `%%bigquery` cells below. The graph will render as an interactive visualization.

> **Reference**: [Knowledge Graph Demo by Google Cloud](https://github.com/GoogleCloudPlatform/devrel-demos/blob/main/data-analytics/knowledge_graph_demo/kg_demo_template.ipynb)

In [16]:
# ============================================================
# BQ Studio: Create Property Graph
# Uncomment the %%bigquery magic below when running in BQ Studio
# ============================================================

# %%bigquery
# CREATE OR REPLACE PROPERTY GRAPH `{DATASET_ID}.adcp_context_graph`
#   NODE TABLES (
#     `{DATASET_ID}.{TABLE_ID}` AS TechNode
#       KEY (span_id)
#       LABEL TechNode
#       PROPERTIES (event_type, agent, timestamp, session_id, invocation_id,
#                   content, latency_ms, status, error_message),
#     `{DATASET_ID}.adcp_biz_nodes` AS BizNode
#       KEY (node_value)
#       LABEL BizNode
#       PROPERTIES (node_type, node_value, confidence, session_id)
#   )
#   EDGE TABLES (
#     `{DATASET_ID}.{TABLE_ID}` AS Caused
#       KEY (span_id)
#       SOURCE KEY (parent_span_id) REFERENCES TechNode (span_id)
#       DESTINATION KEY (span_id) REFERENCES TechNode (span_id)
#       LABEL Caused,
#     `{DATASET_ID}.adcp_cross_links` AS Evaluated
#       KEY (span_id)
#       SOURCE KEY (span_id) REFERENCES TechNode (span_id)
#       DESTINATION KEY (node_value) REFERENCES BizNode (node_value)
#       LABEL Evaluated
#   )

# Print the DDL for copy-paste into BQ Studio
print("Copy the DDL from the cell above into a %%bigquery cell in BQ Studio.")
print("The Property Graph DDL was generated by the SDK in the previous cell.")

Copy the DDL from the cell above into a %%bigquery cell in BQ Studio.
The Property Graph DDL was generated by the SDK in the previous cell.


---

## Phase 7: GQL Reasoning Chain Traversal

With the Property Graph in place, we use **Graph Query Language (GQL)** to answer the question:

> _"Why was the Yahoo Homepage selected for the $50K ELF campaign?"_

The GQL query traces causal hops from the final decision back to the business inputs that informed it.

In [17]:
# Generate the GQL reasoning chain query
gql_query = cgm.get_reasoning_chain_gql(
    decision_event_type="AGENT_COMPLETED",
    biz_entity="Yahoo Homepage",
    max_hops=15,
)

print("=" * 70)
print("  GQL: Why was Yahoo Homepage selected?")
print("=" * 70)
print(gql_query)

  GQL: Why was Yahoo Homepage selected?
GRAPH `test-project-0728-467323.agent_analytics.adcp_context_graph`
MATCH
  (decision:TechNode)-[c:Caused]->{1,15}(step:TechNode)
    -[e:Evaluated]->(biz:BizNode)
WHERE decision.event_type = @decision_event_type
  AND biz.node_value = 'Yahoo Homepage'
RETURN
  TO_JSON(decision) AS decision_node,
  decision.span_id AS decision_span_id,
  decision.event_type AS decision_type,
  step.span_id AS reasoning_span_id,
  step.event_type AS step_type,
  step.agent AS step_agent,
  COALESCE(
    JSON_EXTRACT_SCALAR(step.content, '$.text_summary'),
    JSON_EXTRACT_SCALAR(step.content, '$.response'),
    ''
  ) AS reasoning_text,
  step.latency_ms AS step_latency_ms,
  biz.node_type AS entity_type,
  biz.node_value AS entity_value,
  biz.confidence AS entity_confidence,
  TO_JSON(step) AS step_node,
  TO_JSON(biz) AS biz_node
ORDER BY step.timestamp ASC
LIMIT @result_limit



In [18]:
# Execute the GQL query (requires Property Graph to be created)
try:
    chain = cgm.explain_decision(
        decision_event_type="AGENT_COMPLETED",
        biz_entity="Yahoo Homepage",
    )
    print(f"Reasoning chain: {len(chain)} steps")
    for step in chain:
        print(f"  [{step.get('step_type', '?')}] "
              f"{step.get('step_agent', '?')}: "
              f"{step.get('reasoning_text', '')[:150]}")
        print(f"    -> Entity: {step.get('entity_type', '?')}: "
              f"{step.get('entity_value', '?')}")
except Exception as exc:
    print(f"GQL traversal: {exc}")
    print("\nThe GQL query above can be run in BigQuery Console")
    print("once the Property Graph is created.")

GQL reasoning chain query failed: 400 Property graphs are not supported.; reason: invalid, message: Property graphs are not supported.

Location: US
Job ID: 87c077bb-8160-44e3-87f1-82adbee95d1d



Reasoning chain: 0 steps


In [19]:
# Full causal chain for the ELF campaign session
causal_gql = cgm.get_causal_chain_gql(
    session_id=session_ids[0],
    max_hops=20,
)

print("=" * 70)
print("  GQL: Full Causal Chain for ELF Campaign")
print("=" * 70)
print(causal_gql)

  GQL: Full Causal Chain for ELF Campaign
GRAPH `test-project-0728-467323.agent_analytics.adcp_context_graph`
MATCH
  (root:TechNode)-[c:Caused]->{1,20}(leaf:TechNode)
WHERE root.session_id = @session_id
  AND root.event_type = 'USER_MESSAGE_RECEIVED'
RETURN
  TO_JSON(root) AS root_node,
  root.span_id AS root_span_id,
  leaf.span_id AS leaf_span_id,
  leaf.event_type AS leaf_event_type,
  leaf.agent AS leaf_agent,
  COALESCE(
    JSON_EXTRACT_SCALAR(leaf.content, '$.text_summary'),
    JSON_EXTRACT_SCALAR(leaf.content, '$.response'),
    ''
  ) AS leaf_content,
  leaf.latency_ms AS leaf_latency_ms,
  TO_JSON(leaf) AS leaf_node,
  TO_JSON(c) AS edge
ORDER BY leaf.timestamp ASC
LIMIT @result_limit



In [20]:
# ============================================================
# BQ Studio: Visualize the Context Graph
# Uncomment the %%bigquery magic below when running in BQ Studio.
# These cells render interactive graph visualizations.
# ============================================================

# --- Visualize ALL relationships in the Context Graph ---
# %%bigquery --graph display_only
# GRAPH `agent_analytics.adcp_context_graph`
# MATCH (source)-[r]->(target)
# RETURN
#   TO_JSON(source) AS Source_Node,
#   TO_JSON(r) AS Edge,
#   TO_JSON(target) AS Target_Node

# --- Reasoning Chain: Why was Yahoo Homepage selected? ---
# %%bigquery --graph display_only
# GRAPH `agent_analytics.adcp_context_graph`
# MATCH (decision:TechNode)-[c:Caused]->{1,15}(step:TechNode)
#       -[e:Evaluated]->(biz:BizNode)
# WHERE decision.event_type = 'AGENT_COMPLETED'
#   AND biz.node_value = 'Yahoo Homepage'
# RETURN
#   TO_JSON(decision) AS decision_node,
#   TO_JSON(c) AS caused_edge,
#   TO_JSON(step) AS reasoning_step,
#   TO_JSON(e) AS evaluated_edge,
#   TO_JSON(biz) AS business_entity

# --- Full Causal Chain for a session ---
# %%bigquery --graph display_only
# GRAPH `agent_analytics.adcp_context_graph`
# MATCH (root:TechNode)-[c:Caused]->{1,20}(leaf:TechNode)
# WHERE root.event_type = 'USER_MESSAGE_RECEIVED'
# RETURN
#   TO_JSON(root) AS root_node,
#   TO_JSON(c) AS caused_edge,
#   TO_JSON(leaf) AS leaf_node

print("GQL visualization cells are ready for BQ Studio.")
print("Uncomment the %%bigquery --graph cells above when running in BQ Studio.")

GQL visualization cells are ready for BQ Studio.
Uncomment the %%bigquery --graph cells above when running in BQ Studio.


---

## Phase 8: World Change Detection

A2A direct deal workflows can pause for **days or weeks** waiting for human approval. The Context Graph might become stale:
- Yahoo Homepage inventory is sold out
- CPM prices have changed
- Target audience segments have shifted

We run a **"diff check"** before the HITL approval is finalized. The SDK traverses the graph to find the original BizNodes, queries current availability, and alerts the manager if the "World State" has drifted.

In [21]:
from bigquery_agent_analytics import WorldChangeReport


def check_current_inventory(biz_node):
    """Simulate checking current inventory state.

    In production, this would call the actual Yahoo inventory API.
    Here we simulate that Yahoo Homepage is now sold out (world changed!)
    but Yahoo Mail is still available.
    """
    if biz_node.node_type != "Product":
        return {"available": True, "current_value": biz_node.node_value}

    if biz_node.node_value == "Yahoo Homepage":
        return {
            "available": False,
            "current_value": "SOLD OUT -- Q2 inventory depleted",
            "drift_type": "inventory_depleted",
            "severity": 0.95,
            "recommendation": (
                "Yahoo Homepage inventory is sold out for Q2. "
                "Consider reallocating budget to Yahoo Finance "
                "(similar premium placement, 2.3M impressions available)."
            ),
        }

    return {"available": True, "current_value": biz_node.node_value}


# Run world change detection on the ELF campaign
# First, store some manual biz nodes for the session to detect against
try:
    report = cgm.detect_world_changes(
        session_id=session_ids[0],
        current_state_fn=check_current_inventory,
    )
    print(report.summary())
except Exception as exc:
    print(f"Note: World change detection requires biz_nodes table: {exc}")
    print("\nDemonstrating with manual nodes instead...")

    # Demonstrate with manual nodes directly
    from bigquery_agent_analytics.context_graph import WorldChangeAlert

    manual_report = WorldChangeReport(
        session_id=session_ids[0],
        alerts=[
            WorldChangeAlert(
                biz_node="Yahoo Homepage",
                original_state="Product: Yahoo Homepage (available, CPM $12.50)",
                current_state="SOLD OUT -- Q2 inventory depleted",
                drift_type="inventory_depleted",
                severity=0.95,
                recommendation=(
                    "Yahoo Homepage inventory is sold out for Q2. "
                    "Consider reallocating $8,000 budget to Yahoo Finance "
                    "(similar premium placement, 2.3M impressions available)."
                ),
            ),
        ],
        total_entities_checked=7,
        stale_entities=1,
        is_safe_to_approve=False,
    )
    print(manual_report.summary())
    print(f"\nRecommendation: {manual_report.alerts[0].recommendation}")

World Change Report — Session: adcp-a20d176b82af
  Entities checked : 67
  Stale entities   : 4
  Safe to approve  : False
  [inventory_depleted] Yahoo Homepage: Product: Yahoo Homepage -> SOLD OUT -- Q2 inventory depleted (severity=0.95)
  [inventory_depleted] Yahoo Homepage: Product: Yahoo Homepage -> SOLD OUT -- Q2 inventory depleted (severity=0.95)
  [inventory_depleted] Yahoo Homepage: Product: Yahoo Homepage -> SOLD OUT -- Q2 inventory depleted (severity=0.95)
  [inventory_depleted] Yahoo Homepage: Product: Yahoo Homepage -> SOLD OUT -- Q2 inventory depleted (severity=0.95)


---

## Phase 9: SDK Evaluation Pipeline

Now we evaluate the ADCP agent's performance using the full SDK evaluation stack.

### 9a. Code-Based Evaluation

In [22]:
from bigquery_agent_analytics import CodeEvaluator

trace_filter = TraceFilter(session_ids=session_ids)

presets = [
    ("latency", CodeEvaluator.latency(threshold_ms=30000)),
    ("turn_count", CodeEvaluator.turn_count(max_turns=10)),
    ("error_rate", CodeEvaluator.error_rate(max_error_rate=0.1)),
    ("token_efficiency", CodeEvaluator.token_efficiency(max_tokens=100000)),
]

for name, evaluator in presets:
    try:
        report = asyncio.get_event_loop().run_until_complete(
            asyncio.to_thread(
                client.evaluate,
                evaluator=evaluator,
                filters=trace_filter,
            )
        )
        print(f"\n[{name}]")
        print(report.summary())
    except Exception as exc:
        print(f"\n[{name}] Failed: {exc}")


[latency]
Evaluation Report: latency_evaluator
  Dataset: test-project-0728-467323.agent_analytics.agent_events WHERE session_id IN UNNEST(@session_ids)
  Sessions: 3
  Passed: 3 (100%)
  Failed: 0
  Aggregate Scores:
    latency: 0.842



[turn_count]
Evaluation Report: turn_count_evaluator
  Dataset: test-project-0728-467323.agent_analytics.agent_events WHERE session_id IN UNNEST(@session_ids)
  Sessions: 3
  Passed: 3 (100%)
  Failed: 0
  Aggregate Scores:
    turn_count: 0.867



[error_rate]
Evaluation Report: error_rate_evaluator
  Dataset: test-project-0728-467323.agent_analytics.agent_events WHERE session_id IN UNNEST(@session_ids)
  Sessions: 3
  Passed: 3 (100%)
  Failed: 0
  Aggregate Scores:
    error_rate: 1.000



[token_efficiency]
Evaluation Report: token_efficiency_evaluator
  Dataset: test-project-0728-467323.agent_analytics.agent_events WHERE session_id IN UNNEST(@session_ids)
  Sessions: 3
  Passed: 3 (100%)
  Failed: 0
  Aggregate Scores:
    token_efficiency: 0.932


### 9b. LLM-as-Judge Evaluation

In [23]:
from bigquery_agent_analytics import LLMAsJudge

# Correctness: Does the agent follow ADCP protocol correctly?
judge_correctness = LLMAsJudge.correctness(threshold=0.6)
try:
    report = asyncio.get_event_loop().run_until_complete(
        asyncio.to_thread(
            client.evaluate,
            evaluator=judge_correctness,
            filters=trace_filter,
        )
    )
    print("[LLM Judge: Correctness]")
    print(report.summary())
    print("\nPer-session details:")
    for ss in report.session_scores:
        print(f"  {ss.session_id}: scores={ss.scores} "
              f"passed={ss.passed}")
        if ss.llm_feedback:
            print(f"    Feedback: {ss.llm_feedback[:200]}")
except Exception as exc:
    print(f"Correctness judge failed: {exc}")

[LLM Judge: Correctness]
Evaluation Report: correctness_judge
  Dataset: test-project-0728-467323.agent_analytics.agent_events WHERE session_id IN UNNEST(@session_ids)
  Sessions: 3
  Passed: 3 (100%)
  Failed: 0
  Aggregate Scores:
    correctness: 0.900

Per-session details:
  adcp-2c401a645c40: scores={'correctness': 1.0} passed=True
    Feedback: correctness: The agent provided a comprehensive and accurate response that directly addresses the user's media buying brief. It correctly allocated the budget, provided plausible projections for impre
  adcp-7d9855e7a71b: scores={'correctness': 0.7} passed=True
    Feedback: correctness: The agent provides a comprehensive and well-structured media plan with correct budget calculations. The strategic reasoning is logical. However, the agent assumes flight dates (June 1, 20
  adcp-a20d176b82af: scores={'correctness': 1.0} passed=True
    Feedback: correctness: The agent correctly identified the need to provision the campaign in Google Ad Man

### 9c. Trajectory Matching -- ADCP Workflow Compliance

In [24]:
from bigquery_agent_analytics import BigQueryTraceEvaluator
from bigquery_agent_analytics.trace_evaluator import MatchType

trace_evaluator = BigQueryTraceEvaluator(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
)

# Golden trajectory for the full ADCP workflow
golden_adcp_full = [
    {"tool_name": "query_ad_inventory"},
    {"tool_name": "match_target_audience"},
    {"tool_name": "allocate_media_budget"},
    {"tool_name": "provision_campaign_in_gam"},
]

# Golden trajectory for brief processing only (no provisioning)
golden_adcp_brief = [
    {"tool_name": "query_ad_inventory"},
    {"tool_name": "match_target_audience"},
    {"tool_name": "allocate_media_budget"},
]

# Evaluate ELF campaign (full workflow with provisioning)
try:
    result = asyncio.get_event_loop().run_until_complete(
        trace_evaluator.evaluate_session(
            session_id=session_ids[0],
            golden_trajectory=golden_adcp_full,
            match_type=MatchType.IN_ORDER,
        )
    )
    print("[Trajectory: ELF Campaign -- Full ADCP Workflow (IN_ORDER)]")
    print(f"  Session : {result.session_id}")
    print(f"  Status  : {result.eval_status}")
    print(f"  Scores  : {result.scores}")
except Exception as exc:
    print(f"Trajectory evaluation failed: {exc}")

[Trajectory: ELF Campaign -- Full ADCP Workflow (IN_ORDER)]
  Session : adcp-a20d176b82af
  Status  : EvalStatus.PASSED
  Scores  : {'trajectory_in_order': 1.0, 'step_efficiency': 0.8}


In [25]:
# Batch trajectory evaluation across all sessions
eval_dataset = [
    {
        "session_id": session_ids[0],
        "expected_trajectory": golden_adcp_full,
    },
    {
        "session_id": session_ids[1],
        "expected_trajectory": golden_adcp_brief,
    },
    {
        "session_id": session_ids[2],
        "expected_trajectory": golden_adcp_full,
    },
]

try:
    batch_results = asyncio.get_event_loop().run_until_complete(
        trace_evaluator.evaluate_batch(
            eval_dataset=eval_dataset,
            match_type=MatchType.ANY_ORDER,
        )
    )
    print("[Batch Trajectory Evaluation -- ANY_ORDER]")
    for r in batch_results:
        print(f"  {r.session_id}: {r.eval_status}  "
              f"scores={r.scores}")
except Exception as exc:
    print(f"Batch evaluation failed: {exc}")

[Batch Trajectory Evaluation -- ANY_ORDER]
  adcp-a20d176b82af: EvalStatus.PASSED  scores={'trajectory_any_order': 1.0, 'step_efficiency': 0.8}
  adcp-7d9855e7a71b: EvalStatus.PASSED  scores={'trajectory_any_order': 1.0, 'step_efficiency': 0.6}
  adcp-2c401a645c40: EvalStatus.PASSED  scores={'trajectory_any_order': 1.0, 'step_efficiency': 0.8}


### 9d. Grader Pipeline -- Composite ADCP Quality Score

In [26]:
import contextlib
import io

from bigquery_agent_analytics import (
    GraderPipeline,
    WeightedStrategy,
)

pipeline = (
    GraderPipeline(WeightedStrategy(threshold=0.6))
    .add_code_grader(
        CodeEvaluator.latency(threshold_ms=30000),
        weight=1.0,
    )
    .add_code_grader(
        CodeEvaluator.error_rate(max_error_rate=0.1),
        weight=1.0,
    )
    .add_llm_grader(
        LLMAsJudge.correctness(threshold=0.6),
        weight=2.0,
    )
)

# Evaluate the ELF campaign trace
trace_idx = 0
if traces[trace_idx] is not None:
    trace = traces[trace_idx]
    session_summary = {
        "session_id": trace.session_id,
        "total_events": len(trace.spans),
        "tool_calls": len(trace.tool_calls),
        "tool_errors": len(trace.error_spans),
        "llm_calls": sum(
            1 for s in trace.spans
            if s.event_type in ("llm_request", "llm_response")
        ),
        "avg_latency_ms": (
            trace.total_latency_ms / max(len(trace.spans), 1)
            if trace.total_latency_ms
            else 0.0
        ),
        "max_latency_ms": max(
            (s.latency_ms or 0 for s in trace.spans), default=0
        ),
        "total_latency_ms": trace.total_latency_ms or 0.0,
        "turn_count": sum(
            1 for s in trace.spans if s.event_type == "user_message"
        ),
        "has_error": len(trace.error_spans) > 0,
        "input_tokens": sum(
            s.attributes.get("input_tokens", 0) or 0
            for s in trace.spans
        ),
        "output_tokens": sum(
            s.attributes.get("output_tokens", 0) or 0
            for s in trace.spans
        ),
        "total_tokens": sum(
            s.attributes.get("total_tokens", 0) or 0
            for s in trace.spans
        ),
    }

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        trace_text = trace.render(format="tree")
    if not isinstance(trace_text, str):
        trace_text = buf.getvalue()
    final_response = trace.final_response or ""

    verdict = asyncio.get_event_loop().run_until_complete(
        pipeline.evaluate(
            session_summary=session_summary,
            trace_text=trace_text,
            final_response=final_response,
        )
    )

    print("[GraderPipeline -- ADCP Quality Score]")
    print(f"  Final score : {verdict.final_score:.3f}")
    print(f"  Passed      : {verdict.passed}")
    print(f"  Strategy    : {verdict.strategy_name}")
    print(f"  Grader breakdown:")
    for gr in verdict.grader_results:
        print(f"    - {gr.grader_name}: scores={gr.scores} "
              f"passed={gr.passed}")
else:
    print("Trace not available -- skipping pipeline evaluation.")

[GraderPipeline -- ADCP Quality Score]
  Final score : 0.991
  Passed      : True
  Strategy    : weighted
  Grader breakdown:
    - latency_evaluator: scores={'latency': 0.97443249375} passed=True
    - error_rate_evaluator: scores={'error_rate': 1.0} passed=True
    - correctness_judge: scores={'correctness': 1.0} passed=True


---

## Phase 10: Eval Suite & Multi-Trial

Define a reusable **EvalSuite** for ADCP workflow compliance and run multi-trial evaluation.

In [27]:
from bigquery_agent_analytics import (
    EvalSuite,
    EvalTaskDef,
    EvalCategory,
    EvalValidator,
)

suite = EvalSuite(name="adcp_workflow_evals")

suite.add_task(EvalTaskDef(
    task_id="elf_full_workflow",
    session_id=session_ids[0],
    description="ELF Cosmetics full ADCP workflow: brief -> plan -> provision.",
    category=EvalCategory.CAPABILITY,
    expected_trajectory=golden_adcp_full,
    thresholds={"trajectory_match": 0.8, "latency": 0.7},
    tags=["full_workflow", "brand_awareness"],
))

suite.add_task(EvalTaskDef(
    task_id="nike_brief_processing",
    session_id=session_ids[1],
    description="Nike brief processing: inventory + audience + budget.",
    category=EvalCategory.CAPABILITY,
    expected_trajectory=golden_adcp_brief,
    thresholds={"trajectory_match": 0.9},
    tags=["brief_only", "product_launch"],
))

suite.add_task(EvalTaskDef(
    task_id="tesla_end_to_end",
    session_id=session_ids[2],
    description="Tesla end-to-end: brief -> provision in single turn.",
    category=EvalCategory.REGRESSION,
    expected_trajectory=golden_adcp_full,
    thresholds={"trajectory_match": 0.8},
    tags=["full_workflow", "lead_gen"],
))

print(f"EvalSuite '{suite.name}' -- {len(suite.get_tasks())} tasks:")
for t in suite.get_tasks():
    print(f"  [{t.category.value}] {t.task_id}: {t.description}")

# Validate
warnings = EvalValidator.validate_suite(suite)
print(f"\nValidation: {len(warnings)} warnings")
for w in warnings:
    print(f"  [{w.severity}] {w.check_name}: {w.message}")
if not warnings:
    print("  Suite looks healthy!")

EvalSuite 'adcp_workflow_evals' -- 3 tasks:
  [capability] elf_full_workflow: ELF Cosmetics full ADCP workflow: brief -> plan -> provision.
  [capability] nike_brief_processing: Nike brief processing: inventory + audience + budget.
  [regression] tesla_end_to_end: Tesla end-to-end: brief -> provision in single turn.

Validation: 1 warnings
  [warning] balance: High positive case ratio (100%). Consider adding more negative test cases.


In [28]:
from bigquery_agent_analytics import TrialRunner

trial_runner = TrialRunner(
    evaluator=trace_evaluator,
    num_trials=3,
    concurrency=3,
)

try:
    trial_report = asyncio.get_event_loop().run_until_complete(
        trial_runner.run_trials(
            session_id=session_ids[0],
            golden_trajectory=golden_adcp_full,
            match_type=MatchType.IN_ORDER,
            use_llm_judge=True,
        )
    )
    print("[Multi-Trial -- ELF Campaign, 3 trials]")
    print(f"  pass@k             : {trial_report.pass_at_k:.3f}")
    print(f"  pass^k             : {trial_report.pass_pow_k:.3f}")
    print(f"  per_trial_pass_rate: {trial_report.per_trial_pass_rate:.3f}")
    print(f"  mean_scores        : {trial_report.mean_scores}")
    print(f"\n  Per-trial results:")
    for tr in trial_report.trial_results:
        print(f"    Trial {tr.trial_index}: passed={tr.passed} "
              f"scores={tr.scores}")
except Exception as exc:
    print(f"Multi-trial evaluation failed: {exc}")

[Multi-Trial -- ELF Campaign, 3 trials]
  pass@k             : 1.000
  pass^k             : 1.000
  per_trial_pass_rate: 1.000
  mean_scores        : {'llm_judge_efficiency': 0.8, 'llm_judge_reasoning': 0.26666666666666666, 'llm_judge_task_completion': 0.9666666666666667, 'llm_judge_tool_usage': 0.9333333333333333, 'step_efficiency': 0.8, 'trajectory_in_order': 1.0}

  Per-trial results:
    Trial 0: passed=True scores={'trajectory_in_order': 1.0, 'step_efficiency': 0.8, 'llm_judge_task_completion': 1.0, 'llm_judge_efficiency': 0.8, 'llm_judge_tool_usage': 0.9}
    Trial 1: passed=True scores={'trajectory_in_order': 1.0, 'step_efficiency': 0.8, 'llm_judge_task_completion': 1.0, 'llm_judge_efficiency': 0.9, 'llm_judge_tool_usage': 1.0}
    Trial 2: passed=True scores={'trajectory_in_order': 1.0, 'step_efficiency': 0.8, 'llm_judge_task_completion': 0.9, 'llm_judge_efficiency': 0.7, 'llm_judge_tool_usage': 0.9, 'llm_judge_reasoning': 0.8}


---

## Phase 11: AI-Powered Insights Report

Generate a comprehensive insights report analyzing all ADCP sessions.

In [29]:
import time as _time
from bigquery_agent_analytics import InsightsConfig


def _run_with_retry(fn, max_retries=3, base_delay=5.0):
    """Run a function with exponential backoff on 429/RESOURCE_EXHAUSTED."""
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except Exception as exc:
            is_rate_limit = "429" in str(exc) or "RESOURCE_EXHAUSTED" in str(exc)
            if is_rate_limit and attempt < max_retries:
                delay = base_delay * (2 ** attempt)
                print(f"  Rate limited (attempt {attempt + 1}/{max_retries + 1}),"
                      f" retrying in {delay:.0f}s ...")
                _time.sleep(delay)
            else:
                raise


try:
    insights_report = _run_with_retry(
        lambda: client.insights(
            filters=TraceFilter(session_ids=session_ids),
            config=InsightsConfig(
                max_sessions=10,
                min_events_per_session=3,
                min_turns_per_session=1,
            ),
        ),
        max_retries=3,
        base_delay=5.0,
    )
    print("[ADCP Insights Report]")
    print(insights_report.summary())
except Exception as exc:
    print(f"Insights generation failed after retries: {exc}")

[ADCP Insights Report]
Agent Insights Report
  Generated: 2026-03-05 09:31 UTC
  Sessions analyzed: 3
  Success rate: 100%
  Avg effectiveness: 10.0/10
  Avg latency: 4737ms
  Avg turns: 1.3
  Error rate: 0.0%

  Top Goals:
    data_retrieval: 3
    planning: 3
    analysis: 2
    task_automation: 2
  Outcomes:
    success: 3

  Analysis Sections:
    - Task Areas
    - Interaction Patterns
    - What Works Well
    - Friction Analysis
    - Tool Usage Patterns
    - Improvement Suggestions
    - Trends & Anomalies


In [30]:
# Executive summary (with retry backoff)
try:
    if insights_report.executive_summary:
        print("[Executive Summary]")
        print(insights_report.executive_summary)
    else:
        # Generate if not yet available
        exec_summary = _run_with_retry(
            lambda: insights_report.generate_executive_summary(),
            max_retries=3,
            base_delay=5.0,
        )
        print("[Executive Summary]")
        print(exec_summary)
except NameError:
    print("Insights report not available -- run previous cell first.")
except Exception as exc:
    if "429" in str(exc) or "RESOURCE_EXHAUSTED" in str(exc):
        print(f"Executive summary failed: {exc}")
        print("The insights report was generated successfully above.")
        print("Executive summary can be retried after the rate limit resets.")
    else:
        print(f"Executive summary failed: {exc}")

[Executive Summary]
The agent demonstrates exceptional performance, achieving a 100% success rate and perfect effectiveness with high user satisfaction across all interactions. Users primarily leverage it for critical campaign management tasks, including media planning, data retrieval, and provisioning, often through short, task-oriented sessions. The most notable friction point is a significant average latency of 4.7 seconds per interaction, which could impact user perception despite successful task completion. While functionally flawless, implementing advanced user feedback and sentiment analysis is crucial to uncover subtle user experience issues. Prioritizing latency reduction and gaining deeper qualitative insights will further enhance this highly reliable agent's value and user delight.


In [31]:
# Per-session facets
try:
    print("[Session Facets]")
    for facet in insights_report.session_facets:
        print(f"\n  Session: {facet.session_id}")
        if facet.goal_categories:
            print(f"    Goal categories  : {facet.goal_categories}")
        if facet.outcome:
            print(f"    Outcome          : {facet.outcome}")
        if facet.key_topics:
            print(f"    Key topics       : {facet.key_topics}")
        print(f"    Effectiveness    : {facet.agent_effectiveness}")
except NameError:
    print("Insights report not available -- run previous cells first.")

[Session Facets]

  Session: adcp-a20d176b82af
    Goal categories  : ['data_retrieval', 'analysis', 'planning', 'task_automation']
    Outcome          : success
    Key topics       : ['Media planning', 'Campaign provisioning', 'Ad inventory']
    Effectiveness    : 10.0

  Session: adcp-7d9855e7a71b
    Goal categories  : ['data_retrieval', 'analysis', 'planning']
    Outcome          : success
    Key topics       : ['media buying', 'campaign planning', 'budget allocation']
    Effectiveness    : 10.0

  Session: adcp-2c401a645c40
    Goal categories  : ['data_retrieval', 'planning', 'task_automation']
    Outcome          : success
    Key topics       : ['media buying', 'campaign provisioning', 'ad inventory']
    Effectiveness    : 10.0


---

## Phase 12: End-to-End Pipeline (One-Shot)

The `build_context_graph()` method runs the full pipeline in a single call: extract entities, create cross-links, and create the Property Graph.

In [32]:
# One-shot pipeline: extract + cross-link + create graph
try:
    results = cgm.build_context_graph(
        session_ids=session_ids,
        use_ai_generate=True,
    )
    print("[Context Graph Pipeline Results]")
    print(f"  Biz nodes extracted    : {results['biz_nodes_count']}")
    print(f"  Cross-links created    : {results['cross_links_created']}")
    print(f"  Property Graph created : {results['property_graph_created']}")
except Exception as exc:
    print(f"Pipeline: {exc}")
    print("\nThe individual steps (extract, cross-link, create graph)")
    print("can be run separately as shown in Phases 5-7.")

Failed to create Property Graph: 400 Unsupported statement CREATE PROPERTY GRAPH.; reason: invalidQuery, message: Unsupported statement CREATE PROPERTY GRAPH.

Location: US
Job ID: 7dbcf5d1-12f1-4d97-927d-0af5ed51b2a4



[Context Graph Pipeline Results]
  Biz nodes extracted    : 179
  Cross-links created    : True
  Property Graph created : False


---

## Summary

This notebook demonstrated the **Context Graph V2: System of Reasoning** for agentic advertising:

| Phase | Feature | Description |
|---|---|---|
| 1 | **ADCP Tools** | Deterministic Yahoo inventory, audience, budget, and GAM provisioning tools |
| 2 | **Multi-Agent System** | Yahoo Sales Agent with ADCP workflow orchestration |
| 3 | **Trace Logging** | BigQueryAgentAnalyticsPlugin captures every event |
| 4 | **Trace Visualization** | SDK Client retrieves and renders hierarchical execution DAGs |
| 5 | **Biz Entity Extraction** | AI.GENERATE extracts Products, Targeting, Campaigns from payloads |
| 6 | **Property Graph DDL** | CREATE PROPERTY GRAPH with TechNode, BizNode, Caused, Evaluated |
| 7 | **GQL Traversal** | Quantified-path queries: "Why was Yahoo Homepage selected?" |
| 8 | **World Change Detection** | Temporal drift check before HITL approval |
| 9 | **Full Evaluation** | Code metrics, LLM judge, trajectory matching, grader pipeline |
| 10 | **Eval Suite** | Reusable ADCP compliance tests with multi-trial pass@k |
| 11 | **AI Insights** | Multi-stage analysis with executive summary |
| 12 | **One-Shot Pipeline** | `build_context_graph()` runs extract -> cross-link -> create |

### Key Takeaways

- **System of Reasoning**: The Context Graph cross-links "what the agent did" (TechGraph) with "why it did it" (BizGraph), providing the automated explainability required for HITL ad planners to trust multi-thousand-dollar media decisions.
- **Native Property Graphs**: BigQuery's `CREATE PROPERTY GRAPH` + GQL replaces cumbersome recursive CTEs with elegant graph traversal.
- **World Change Detection**: Long-running A2A tasks (days/weeks) need temporal intelligence to detect stale context before final HITL approval.
- **End-to-End Observability**: From agent execution to business entity extraction to evaluation -- all powered by the BigQuery Agent Analytics SDK.

In [33]:
# Cleanup
try:
    asyncio.get_event_loop().run_until_complete(
        plugin.shutdown(timeout=10.0)
    )
except Exception:
    pass

print("\nDemo complete!")
print(f"Sessions: {session_ids}")
print(f"Traces logged to: {PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")
print(f"Context Graph: {PROJECT_ID}.{DATASET_ID}.{cg_config.graph_name}")


Demo complete!
Sessions: ['adcp-a20d176b82af', 'adcp-7d9855e7a71b', 'adcp-2c401a645c40']
Traces logged to: test-project-0728-467323.agent_analytics.agent_events
Context Graph: test-project-0728-467323.agent_analytics.adcp_context_graph
